In [ ]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
from cleanco import basename
from china_cities import cities as city_names
import yaml

In [ ]:
import os
os.chdir('../../../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Load data from the dataset

In [ ]:
patent_data = pd.read_csv(dataset_config['path_uspat'] + 'g_assignee_disambiguated.tsv', sep='\t', usecols=['patent_id', 'assignee_sequence', 'disambig_assignee_organization', 'assignee_type'])
patent_data

In [ ]:
patents_org = patent_data.disambig_assignee_organization.dropna().drop_duplicates()
patents_org

In [ ]:
firm_data = pd.read_csv(dataset_config['path_processed'] + 'CN_CN/EL_website_cleaned.csv')
firm_data

In [ ]:
firm = firm_data.EL_name.dropna().drop_duplicates()
firm

## Unify company names by removing company suffixes

In [ ]:
'''
def clean_company_suffix(name):
    import re
    from cleanco import basename
    name_dedot = re.sub('[^\w\s]', '', name)
    name_clean = re.sub(' LLC$| Limited| Company| holding| liability| Group| Office| Corporation| Corp$| GmbH$| GmbH \& Co$| Co$| Co ltd| Ltd$| Inc$| AG$| NV$| Spa$| Srl$| PLC$| AB$| BV$| LP$| SAS$| OY$| Mfy$| SA$| Sdn Bhd$| Bhd$| Pty Ltd$| Pte Ltd$', '', name_dedot, flags=re.IGNORECASE)
    name_joined = ' '.join(re.split('\s+', name_clean)).strip().lower()
    
    return basename(name_joined)
'''

In [ ]:
_SUFFIX_RE = re.compile(
    r"(?:\s+("
    r"LLC|Limited|Company|Holding|Liability|Group|Office|Corporation|Corp|"
    r"GmbH(?:\s*&\s*Co)?|Co|Co\s*ltd|Ltd|Inc|AG|NV|Spa|Srl|PLC|AB|BV|LP|SAS|"
    r"OY|Mfy|SA|Sdn\s*Bhd|Bhd|Pty\s*Ltd|Pte\s*Ltd"
    r"))\s*$",
    flags=re.IGNORECASE,
)

def clean_company_suffix(name: str) -> str:
    name_dedot = re.sub(r"[^\w\s]", "", name)
    name_clean = _SUFFIX_RE.sub("", name_dedot)
    name_joined = " ".join(re.split(r"\s+", name_clean)).strip().lower()
    return basename(name_joined)

In [ ]:
# test
clean_company_suffix('123.ddd.cc,  LLc')

In [ ]:
# (1) clean patent's applicants
patents_org_cleaned = patents_org.apply(clean_company_suffix)
patents_org_cleaned

In [ ]:
# (2) clean firms
firm_cleaned = firm.apply(clean_company_suffix)
firm_cleaned

In [ ]:
# Build once at import time
_CITY_TERMS = city_names.get_cities_en() + city_names.get_provinces()
_CITY_TERMS = sorted({t.strip() for t in _CITY_TERMS if t and t.strip()}, key=len, reverse=True)

_CITY_RE = re.compile(
    r"\b(?:%s)\b" % "|".join(map(re.escape, _CITY_TERMS)),
    flags=re.IGNORECASE,
)

_WS_RE = re.compile(r"\s+")

def remove_city_names(name: str) -> str:
    if not name:
        return ""
    name_wo_city = _CITY_RE.sub("", name)
    return _WS_RE.sub(" ", name_wo_city).strip()

In [ ]:
'''
def remove_city_names(name):
    import re
    from china_cities import cities as city_names
    city_re = '|'.join(city_names.get_cities_en() + city_names.get_provinces()).lower()
    name_wo_city = re.sub(city_re, '', name)
    name_cleaned = re.sub('\s+', ' ', name_wo_city).strip()
    return name_cleaned
'''

In [ ]:
firm_cleaned = firm_cleaned.apply(remove_city_names)
firm_cleaned

## Generate two-gram form of dataframe

In [ ]:
def two_gram(s):
    L = s.split()
    result = []
    for index in range(len(L) - 1):
        result.append('#' + L[index] + ' ' + L[index + 1] + '#')
    return result

In [ ]:
patents_tg = patents_org_cleaned.apply(two_gram)

In [ ]:
firm_tg = firm_cleaned.apply(two_gram)

## Generate a dictionary of two-gram word list

In [ ]:
def make_word_list(data):
    result = []

    for index in range(data.size):
        for item in data.iloc[index]:
            result.append(item)
    
    return list(set(result))

In [ ]:
all_names_patents = make_word_list(patents_tg)
len(all_names_patents)

In [ ]:
all_names_firm = make_word_list(firm_tg)
len(all_names_firm)

In [ ]:
all_two_gram_words = list(set(all_names_patents + all_names_firm))
len(all_two_gram_words)

## Make index of the search-space dictionary

In [ ]:
def make_index_dict(L):
    result = {}
    for index in range(len(L)):
        result[L[index]] = index

    return result

In [ ]:
index_dict = make_index_dict(all_two_gram_words)
len(index_dict)

In [ ]:
patents_tgn = patents_tg.apply(lambda x: [index_dict[item] for item in x])

In [ ]:
firm_tgn = firm_tg.apply(lambda x: [index_dict[item] for item in x])

In [ ]:
def make_pair_loc_dict(data):
    result = {}
    for index in tqdm(range(data.size)):
        for item in data.iloc[index]:
            if item in result:
                result[item].append(index)
            else:
                result[item] = [index]
    return result

In [ ]:
pair_loc_dict = make_pair_loc_dict(firm_tgn)

## Locate and record matches

In [ ]:
def find_matches(name, cutoff=1):
    if len(name) == 0:
        return None
    def single_match(query, data, cutoff):
        if len(data) - len(query) > cutoff:
            return False
        for idx in range(min(len(query), len(data))):
            if query[idx] != data[idx]:
                return False
        return True
    
    search_index = []
    for item in name:
        if item in pair_loc_dict:
            search_index += pair_loc_dict[item]
    search_index = list(set(search_index))
    #print(search_index)

    result = firm_tgn.iloc[search_index].apply(lambda x: single_match(name , x, cutoff))
    if result.sum():
        #print(names_firm_two_gram)
        #print(result)
        return result[result].index[0]
    else:
        return None


In [ ]:
matches = []
matches

prev_checkpoint = -1
START_IDX = 0

for index in range(START_IDX, patents_tgn.size):
    result = find_matches(patents_tgn.iloc[index])
    if result:
        matches.append((index, result))
    if (1 + len(matches)) % 1000 == 0 and prev_checkpoint != len(matches):
        print('Checkpoint saved:', len(matches), '\t', 'Current index:', index)
        prev_checkpoint = len(matches)

## Output matches in the original name format (from raw data)

In [ ]:
def matches_metadata_output(matches):
    for idx_0, idx_1 in matches:
        print(patents_tgn.iloc[idx_0], firm_tgn[idx_1], sep='\n')
        print()

In [ ]:
matches_metadata_output(matches)

## Generate a dataframe of matches and inner-join with raw data

In [ ]:
def gen_match_df(matches):
    result1 = []
    result2 = []
    
    for idx_0, idx_1 in matches:
        result1.append(patents_org.iloc[idx_0])
        result2.append(firm[idx_1])

    return pd.DataFrame({'disambig_assignee_organization': result1, 'EL_name': result2})

In [ ]:
df_matches = gen_match_df(matches)
df_matches

In [ ]:
result = pd.merge(patent_data, df_matches, on='disambig_assignee_organization')
final = pd.merge(result, firm_data, on='EL_name', how='left')
final

In [ ]:
final.to_csv(dataset_config['path_processed'] + 'CN_CN/USpatents_EL_matches.csv', index=False)